# Download Transactions
## Written without Claude
Author: **Elisa Warner**

In [81]:
from config import ETHERSCANIO_KEY
import requests
from pathlib import Path
import json
import pickle as pkl

In [16]:
CACHE_FILE = "cache.json"

In [21]:
class Session(object):
    """
    Session class helps us to connect online

    Methods:
     - load_cache : load website cache for curl downloads
     - call_site_curl : download website html with requests package (good for simple sites)
     - get_call_url : combines query parameters with the base url to get exact website address
     - call_site_broweser : selenium-based driver loads website (for more sophisticated sites)
     - save_cache : save the cache for requests-based downloads
    """
    def __init__(self, cache_file = CACHE_FILE):
        self.cache_file = CACHE_FILE
        self.cache = self.load_cache()
        
    def load_cache(self):
        if Path(self.cache_file).exists():
            with open(self.cache_file) as f:
                temp = f.read()
                cache = json.loads(temp)
                print("Load cache from memory")
        else:
            cache = {}
            print("Create new cache")
        return cache
        
    def call_site_curl(self, base_url, params={}):
        call_url = self.get_call_url(base_url, params)
        
        if call_url not in self.cache:
            response = json.loads(requests.get(call_url).text)
            self.cache[call_url] = response
            self.save_cache()
        else:
            print("Loading from memory...")
            response = self.cache[call_url]
        
        return response

    def get_call_url(self, base_url, params={}):
        call_url = base_url
        if params:
            call_url = base_url + "?"
            for p in params:
                call_url += str(p) + "=" + str(params[p])
                if p != list(params)[-1]:
                    call_url += "&"
        print("Calling:", call_url)
        return call_url
        
    def save_cache(self):
        if self.cache:
            with open(self.cache_file, "wb") as f:
                cache_str = json.dumps(self.cache)
                f.write(cache_str.encode())

In [22]:
# load cache
session = Session()

Load cache from memory


In [78]:
## normal
base_url = "https://api.routescan.io/v2/network/mainnet/evm/1/etherscan/api"
shelf_params = [{"action":"txlist"}, 
                {"action":"txlistinternal"},
                {"action":"tokentx"}]

obscure_params = [{"action":"tokentx", "contract_address":"0x4921bB864dE2E557939B074be20Ff4B98723b86b"}, # trumps war
                 {"action":"tokentx", "contract_address":"0x496a35a65C00B4AED125D19df3871e6b4cB05188"}, # trump rekt
                 {"action":"tokentx", "contract_address":"0xEF70241b3cEEB61F437D30816734bfC2c506AB81"}, #chimpxai
                 {"action":"tokentx", "contract_address":"0x1Ff22Cb1c7804b3529A005DdcF9bDE6E7c9D6d90"}, #flying tulip
                 {"action":"tokentx", "contract_address":"0x93fb4459F48f379BA865eb55dDDc292C702ff569"},
                 {"action":"tokentx", "contract_address":"0xdAC17F958D2ee523a2206206994597C13D831ec7"}, # usdt
                 {"action":"tokentx", "contract_address":"0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"}, # usdc
                 {"action":"tokentx", "contract_address":"0x2b591e99afE9f32eAA6214f7B7629768c40Eeb39"}, # hex
                 {"action":"tokentx", "contract_address":"0xD9a3679764855E03b6030db48aE36CEA8a2a7526"}] #trump dog

names = ['normal',
         'internal',
         'erc-20-tokens']


In [83]:
with open("addresses.pkl","rb") as f:
    addresses = pkl.load(f)

In [89]:
addresses.index(address)

26

In [ ]:
for address in addresses[26:]:
    for ind, action_param in enumerate(shelf_params):
        page = 0
        results = []
        res = {'result':'start'}
        
        while res['result']:
            params = {
                "module":"account",
                "address": address,
                "startblock": 0,
                "endblock": 99999999,
                "page": page,
                "offset": 100,
                "sort": "asc",
                "apikey": ETHERSCANIO_KEY
            }
            params.update(action_param)
            res = session.call_site_curl(base_url, params)
        
            if res['result']:
                results += res['result']
            
            page += 1
            if page > 20:
                res = {}

            if not res:
                print("BREAKING: NO RESULTS")
                break
        
        print("Done: %s pages" % page)
        df = pd.DataFrame(results)
        Path("library/%s" % address).mkdir(exist_ok=True)
        df.to_csv('library/%s/%s_transactions.csv' % (address, names[ind]), index=False)

Calling: https://api.routescan.io/v2/network/mainnet/evm/1/etherscan/api?module=account&address=0x9430801EBAf509Ad49202aaBc5F5bc6fD8a3Daf8&startblock=0&endblock=99999999&page=0&offset=100&sort=asc&apikey=rs_af2a1765bce8429a347789cd&action=txlist
Loading from memory...
Calling: https://api.routescan.io/v2/network/mainnet/evm/1/etherscan/api?module=account&address=0x9430801EBAf509Ad49202aaBc5F5bc6fD8a3Daf8&startblock=0&endblock=99999999&page=1&offset=100&sort=asc&apikey=rs_af2a1765bce8429a347789cd&action=txlist
Loading from memory...
Calling: https://api.routescan.io/v2/network/mainnet/evm/1/etherscan/api?module=account&address=0x9430801EBAf509Ad49202aaBc5F5bc6fD8a3Daf8&startblock=0&endblock=99999999&page=2&offset=100&sort=asc&apikey=rs_af2a1765bce8429a347789cd&action=txlist
Loading from memory...
Calling: https://api.routescan.io/v2/network/mainnet/evm/1/etherscan/api?module=account&address=0x9430801EBAf509Ad49202aaBc5F5bc6fD8a3Daf8&startblock=0&endblock=99999999&page=3&offset=100&sort=